# 混合检索、融合、重排与评估：一条可回归的工程链路

这份 notebook 不再重复解释“BM25 是什么”，而是回答一个工程问题：**当关键词检索、语义检索、业务规则和 reranker 同时存在时，怎样把它们组成可测量、可降级、可排错的系统？**

## 学习目标

1. 建立 sparse 与 dense 两路候选，并严格执行 tenant/状态过滤。
2. 用 RRF 融合不同量纲的排名，而不是直接相加原始分数。
3. 在较小候选集上加入可解释 reranker，并区分教学特征与生产 cross-encoder。
4. 实现 Recall@k、MRR、nDCG，做候选深度消融。
5. 输出 request trace、延迟预算与降级策略。
6. 用断言覆盖 ACL、旧版本、确定性与引用 id。

本例使用小型受控语料；分数只能验证代码路径，不能外推成真实线上收益。

## 1. 先定义系统契约

在线接口至少接收 `query、tenant_id、filters、top_k、index_version`，返回稳定 `chunk_id、score/rank、source、version、trace_id`。推荐链路：

```text
鉴权/硬过滤
  -> sparse top-N_s + dense top-N_d
  -> 去重与 RRF/校准融合
  -> rerank top-N_r
  -> context budget / citation
  -> 生成或搜索结果
```

`top-N_s`、`top-N_d` 是候选深度，`top-N_r` 是昂贵重排器的输入上限，最终 `top-k` 才是返回数量。把三者混成一个参数，会让质量和延迟无法独立调节。

In [ ]:
from dataclasses import dataclass, asdict  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。
import time  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
from sklearn.feature_extraction.text import TfidfVectorizer  # 导入本单元所需的依赖。
from sklearn.decomposition import TruncatedSVD  # 导入本单元所需的依赖。
from sklearn.preprocessing import normalize  # 导入本单元所需的依赖。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Chunk:  # 定义承载本节状态与行为的数据结构。
    chunk_id: str  # 执行当前语句以推进本节示例。
    title: str  # 执行当前语句以推进本节示例。
    text: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    version: int  # 执行当前语句以推进本节示例。
    active: bool  # 执行当前语句以推进本节示例。
    source: str  # 执行当前语句以推进本节示例。
    authority: float  # 执行当前语句以推进本节示例。

chunks = [  # 计算并保存当前步骤的中间状态。
    Chunk("db-e1042-v2#0", "数据库连接错误 E1042", "E1042 表示数据库连接超时。先检查连接串、网络和连接池。", "company-a", 2, True, "runbook/db", 1.0),  # 执行当前语句以推进本节示例。
    Chunk("login-v1#0", "账号登录失败", "无法登录或忘记密码时，先重置密码，再检查账号是否锁定。", "company-a", 1, True, "help/login", 0.9),  # 执行当前语句以推进本节示例。
    Chunk("travel-v2#0", "上海差旅酒店标准", "2026 年上海出差的酒店报销上限为每晚 650 元。", "company-a", 2, True, "policy/travel", 1.0),  # 执行当前语句以推进本节示例。
    Chunk("travel-v1#0", "旧版上海差旅标准", "旧制度规定上海酒店上限为每晚 500 元。", "company-a", 1, False, "policy/archive", 0.4),  # 执行当前语句以推进本节示例。
    Chunk("refund-v1#0", "退款到账时间", "退款审核通过后通常在 5 个工作日内原路到账。", "company-a", 1, True, "help/refund", 0.9),  # 执行当前语句以推进本节示例。
    Chunk("cancel-v1#0", "取消订单", "未发货订单可在订单详情页取消；已发货订单需申请退货。", "company-a", 1, True, "help/order", 0.9),  # 执行当前语句以推进本节示例。
    Chunk("slow-sql-v1#0", "数据库慢查询", "SQL 查询延迟高时检查执行计划、索引与锁等待。", "company-a", 1, True, "runbook/sql", 0.8),  # 执行当前语句以推进本节示例。
    Chunk("tenant-b-secret#0", "B 公司数据库密钥", "B 公司 E1042 应急密钥是 SECRET-B。", "company-b", 1, True, "private/b", 1.0),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
by_id = {chunk.chunk_id: chunk for chunk in chunks}  # 计算并保存当前步骤的中间状态。
print("chunks:", len(chunks), "active:", sum(c.active for c in chunks))  # 执行当前语句以推进本节示例。


### 1.1 安全边界：tenant 不是用户可随意填写的过滤字符串

生产请求中的 tenant 必须来自服务端验证过的身份声明，而不是 query/body 参数。共享索引上的候选过滤可以防止结果越权，却不能阻止 analyzer 词表、DF/IDF、dense 训练或缓存跨租户混合；高隔离场景应使用 tenant/安全域独立的索引、模型或统计空间。

本教程选择一个明确而保守的合同：后续 sparse 与 dense 索引只用 `company-a` 的 active 文档构建。跨租户密钥和旧版本仍保留在原始 fixture 中，用于证明它们从未进入可检索索引。

In [ ]:
all_chunks = chunks  # 计算并保存当前步骤的中间状态。
INDEX_TENANT = "company-a"  # 计算并保存当前步骤的中间状态。
INDEX_VERSION = "company-a-2026-07"  # 计算并保存当前步骤的中间状态。
chunks = [chunk for chunk in all_chunks if chunk.tenant == INDEX_TENANT and chunk.active]  # 计算并保存当前步骤的中间状态。
by_id = {chunk.chunk_id: chunk for chunk in chunks}  # 计算并保存当前步骤的中间状态。

assert "tenant-b-secret#0" not in by_id  # 用受控断言验证关键不变量。
assert "travel-v1#0" not in by_id  # 用受控断言验证关键不变量。
print("物理索引文档数:", len(chunks), "排除 fixture:", len(all_chunks) - len(chunks))  # 执行当前语句以推进本节示例。


## 2. Analyzer 是检索契约，不是随手清洗

关键词召回是否能命中 `E1042`、金额和领域短语，首先取决于 analyzer。训练/索引与查询必须使用同一版本，并记录词典、大小写、同义词和停用词配置。

下面用小词典最长匹配演示中文；未知汉字退化为单字，ASCII 编号整体保留。生产系统应替换为经过业务评估的 analyzer，并对词典升级做双索引回归。

In [ ]:
LEXICON = sorted({  # 计算并保存当前步骤的中间状态。
    "数据库", "连接", "超时", "连接串", "连接池", "登录", "无法登录", "忘记密码",  # 执行当前语句以推进本节示例。
    "重置密码", "账号", "锁定", "上海", "出差", "酒店", "报销", "上限", "每晚",  # 执行当前语句以推进本节示例。
    "退款", "到账", "审核", "工作日", "取消订单", "订单", "发货", "退货", "慢查询",  # 执行当前语句以推进本节示例。
    "执行计划", "索引", "锁等待", "检查", "应急", "密钥"  # 执行当前语句以推进本节示例。
}, key=len, reverse=True)  # 计算并保存当前步骤的中间状态。

def analyze(text: str) -> list[str]:  # 定义本节可复用的核心函数。
    text = text.lower()  # 计算并保存当前步骤的中间状态。
    tokens, index = [], 0  # 计算并保存当前步骤的中间状态。
    while index < len(text):  # 在终止条件满足前持续推进状态。
        char = text[index]  # 计算并保存当前步骤的中间状态。
        if char.isspace() or char in "，。：；！？、（）()/.":  # 按当前条件选择后续控制路径。
            index += 1  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        ascii_match = re.match(r"[a-z0-9_-]+", text[index:])  # 计算并保存当前步骤的中间状态。
        if ascii_match:  # 按当前条件选择后续控制路径。
            token = ascii_match.group(0)  # 计算并保存当前步骤的中间状态。
            tokens.append(token)  # 执行当前语句以推进本节示例。
            index += len(token)  # 计算并保存当前步骤的中间状态。
            continue  # 调整当前循环或占位控制流。
        matched = next((word for word in LEXICON if text.startswith(word, index)), None)  # 计算并保存当前步骤的中间状态。
        tokens.append(matched or char)  # 执行当前语句以推进本节示例。
        index += len(matched) if matched else 1  # 计算并保存当前步骤的中间状态。
    return tokens  # 返回当前分支计算出的结果。

for sample in ["E1042 怎么处理？", "上海酒店报销上限", "退款多久到账"]:  # 遍历输入元素以累积或检查结果。
    print(sample, "->", analyze(sample))  # 执行当前语句以推进本节示例。


## 3. Sparse 路：BM25 候选

对查询词 $t$ 与文档 $d$，本例使用 Lucene 常见的 IDF 形式：

$$
\operatorname{IDF}(t)=\log\left(1+\frac{N-df_t+0.5}{df_t+0.5}\right),
$$

$$
\operatorname{BM25}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)\frac{tf_{t,d}(k_1+1)}{tf_{t,d}+k_1(1-b+b|d|/\overline{dl})}.
$$

$k_1$ 控制词频饱和，$b$ 控制长度归一化。实现把 title 重复一次作为教学型字段 boost；生产 BM25F 应对各字段分别统计长度并调权。硬 ACL 和 active filter 在 top-N 前执行。

In [ ]:
class BM25Retriever:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, chunks: list[Chunk], k1: float = 1.2, b: float = 0.75):  # 定义本节可复用的核心函数。
        self.chunks, self.k1, self.b = chunks, k1, b  # 计算并保存当前步骤的中间状态。
        self.tokens = [analyze(c.title) * 2 + analyze(c.text) for c in chunks]  # 计算并保存当前步骤的中间状态。
        self.tf = [Counter(row) for row in self.tokens]  # 计算并保存当前步骤的中间状态。
        self.df = Counter()  # 计算并保存当前步骤的中间状态。
        for row in self.tokens:  # 遍历输入元素以累积或检查结果。
            self.df.update(set(row))  # 执行当前语句以推进本节示例。
        self.avgdl = sum(map(len, self.tokens)) / len(self.tokens)  # 计算并保存当前步骤的中间状态。

    def score(self, query: str, row: int) -> float:  # 定义本节可复用的核心函数。
        total, n = 0.0, len(self.chunks)  # 计算并保存当前步骤的中间状态。
        dl, frequencies = len(self.tokens[row]), self.tf[row]  # 计算并保存当前步骤的中间状态。
        for term in analyze(query):  # 遍历输入元素以累积或检查结果。
            tf, df = frequencies.get(term, 0), self.df.get(term, 0)  # 计算并保存当前步骤的中间状态。
            if not tf:  # 按当前条件选择后续控制路径。
                continue  # 调整当前循环或占位控制流。
            idf = math.log(1 + (n - df + 0.5) / (df + 0.5))  # 计算并保存当前步骤的中间状态。
            denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)  # 计算并保存当前步骤的中间状态。
            total += idf * tf * (self.k1 + 1) / denominator  # 计算并保存当前步骤的中间状态。
        return total  # 返回当前分支计算出的结果。

    def search(self, query: str, tenant: str, top_n: int) -> list[tuple[str, float]]:  # 定义本节可复用的核心函数。
        scored = []  # 计算并保存当前步骤的中间状态。
        for row, chunk in enumerate(self.chunks):  # 遍历输入元素以累积或检查结果。
            if chunk.tenant != tenant or not chunk.active:  # 按当前条件选择后续控制路径。
                continue  # 调整当前循环或占位控制流。
            score = self.score(query, row)  # 计算并保存当前步骤的中间状态。
            if score > 0:  # 关键词路不把零命中文档填进 top-N。
                scored.append((chunk.chunk_id, score))  # 执行当前语句以推进本节示例。
        return sorted(scored, key=lambda item: (-item[1], item[0]))[:top_n]  # 返回当前分支计算出的结果。

bm25 = BM25Retriever(chunks)  # 计算并保存当前步骤的中间状态。
print(bm25.search("E1042 怎么处理", "company-a", 4))  # 执行当前语句以推进本节示例。


## 4. Dense 路：用 LSA 演示独立分数空间

生产 dense retriever 通常是双编码器：$u=f_q(q),v=f_d(d)$，再做最大内积或余弦 ANN 搜索。本 notebook 为保持离线可执行，用 TF-IDF + TruncatedSVD 得到低维 LSA 向量；它只是第二路检索器，**不能代表现代 neural embedding 的语义能力**。

关键工程结论不变：dense 分数与 BM25 分数没有天然共同量纲，模型/归一化/索引升级都会改变分布，因此默认不应直接相加。

In [ ]:
texts = [f"{chunk.title} {chunk.text}" for chunk in chunks]  # 计算并保存当前步骤的中间状态。
tfidf = TfidfVectorizer(analyzer=analyze, lowercase=False)  # 计算并保存当前步骤的中间状态。
x_sparse = tfidf.fit_transform(texts)  # 计算并保存当前步骤的中间状态。
n_components = min(5, x_sparse.shape[0] - 1, x_sparse.shape[1] - 1)  # 计算并保存当前步骤的中间状态。
svd = TruncatedSVD(n_components=n_components, random_state=7)  # 计算并保存当前步骤的中间状态。
x_dense = normalize(svd.fit_transform(x_sparse))  # 计算并保存当前步骤的中间状态。

def dense_search(query: str, tenant: str, top_n: int) -> list[tuple[str, float]]:  # 定义本节可复用的核心函数。
    query_vector = normalize(svd.transform(tfidf.transform([query])))[0]  # 计算并保存当前步骤的中间状态。
    scores = x_dense @ query_vector  # 计算并保存当前步骤的中间状态。
    candidates = [  # 计算并保存当前步骤的中间状态。
        (chunk.chunk_id, float(scores[row]))  # 执行当前语句以推进本节示例。
        for row, chunk in enumerate(chunks)  # 遍历输入元素以累积或检查结果。
        if chunk.tenant == tenant and chunk.active  # 按当前条件选择后续控制路径。
    ]  # 执行当前语句以推进本节示例。
    return sorted(candidates, key=lambda item: (-item[1], item[0]))[:top_n]  # 返回当前分支计算出的结果。

print(dense_search("数据库连不上怎么办", "company-a", 4))  # 执行当前语句以推进本节示例。


### 4.1 零向量与最低相关度

ANN 无论问题是否有答案，通常都能返回“最近”的文档；最近不等于相关。教学实现至少拒绝完全不在词表中的零向量和非正相似度，生产系统还要在独立 validation/no-answer 集上选择分桶阈值，并由生成层继续做证据充分性判断。阈值是模型与索引版本的一部分，不能跨版本照搬。

In [ ]:
def dense_search(query: str, tenant: str, top_n: int, min_score: float = 1e-8) -> list[tuple[str, float]]:  # 定义本节可复用的核心函数。
    if tenant != INDEX_TENANT:  # 按当前条件选择后续控制路径。
        return []  # 返回当前分支计算出的结果。
    sparse_query = tfidf.transform([query])  # 计算并保存当前步骤的中间状态。
    if sparse_query.nnz == 0:  # 按当前条件选择后续控制路径。
        return []  # 返回当前分支计算出的结果。
    query_vector = normalize(svd.transform(sparse_query))[0]  # 计算并保存当前步骤的中间状态。
    if not np.any(query_vector):  # 按当前条件选择后续控制路径。
        return []  # 返回当前分支计算出的结果。
    scores = x_dense @ query_vector  # 计算并保存当前步骤的中间状态。
    candidates = [  # 计算并保存当前步骤的中间状态。
        (chunk.chunk_id, float(scores[row]))  # 执行当前语句以推进本节示例。
        for row, chunk in enumerate(chunks)  # 遍历输入元素以累积或检查结果。
        if float(scores[row]) >= min_score  # 按当前条件选择后续控制路径。
    ]  # 执行当前语句以推进本节示例。
    return sorted(candidates, key=lambda item: (-item[1], item[0]))[:top_n]  # 返回当前分支计算出的结果。

assert dense_search("火星量子香蕉", INDEX_TENANT, 4) == []  # 用受控断言验证关键不变量。
print(dense_search("数据库连不上怎么办", INDEX_TENANT, 4))  # 执行当前语句以推进本节示例。


## 5. RRF：先用排名融合建立稳健基线

Reciprocal Rank Fusion 对每个文档累加：

$$
\operatorname{RRF}(d)=\sum_{r\in\mathcal R}\frac{w_r}{c+\operatorname{rank}_r(d)}.
$$

它不要求 BM25 与 dense 分数校准；$c$ 控制头部名次差异，$w_r$ 是可选检索器权重。RRF 不会凭空补回两路都漏掉的证据，所以候选深度必须先用 Recall@N 调。某一路超时或不可用时，融合器应记录降级，而不是静默改变结果。

In [ ]:
def rrf(rankings: dict[str, list[tuple[str, float]]], constant: int = 60, weights=None):  # 定义本节可复用的核心函数。
    weights = weights or {name: 1.0 for name in rankings}  # 计算并保存当前步骤的中间状态。
    scores, trace = defaultdict(float), defaultdict(dict)  # 计算并保存当前步骤的中间状态。
    for name, ranking in rankings.items():  # 遍历输入元素以累积或检查结果。
        for rank, (chunk_id, raw_score) in enumerate(ranking, start=1):  # 遍历输入元素以累积或检查结果。
            contribution = weights[name] / (constant + rank)  # 计算并保存当前步骤的中间状态。
            scores[chunk_id] += contribution  # 计算并保存当前步骤的中间状态。
            trace[chunk_id][name] = {"rank": rank, "raw_score": raw_score, "rrf": contribution}  # 计算并保存当前步骤的中间状态。
    ordered = sorted(scores, key=lambda cid: (-scores[cid], cid))  # 计算并保存当前步骤的中间状态。
    return [(cid, scores[cid]) for cid in ordered], dict(trace)  # 返回当前分支计算出的结果。

def hybrid_candidates(query: str, tenant: str, sparse_n=4, dense_n=4):  # 定义本节可复用的核心函数。
    sparse = bm25.search(query, tenant, sparse_n)  # 计算并保存当前步骤的中间状态。
    dense = dense_search(query, tenant, dense_n)  # 计算并保存当前步骤的中间状态。
    fused, trace = rrf({"bm25": sparse, "dense": dense})  # 计算并保存当前步骤的中间状态。
    return fused, trace  # 返回当前分支计算出的结果。

fused, fusion_trace = hybrid_candidates("数据库连不上怎么办", "company-a")  # 计算并保存当前步骤的中间状态。
print("fused:", fused[:5])  # 执行当前语句以推进本节示例。
print("top trace:", fusion_trace[fused[0][0]])  # 执行当前语句以推进本节示例。


## 6. Reranker 只处理有限候选

bi-encoder/检索器把 query 和文档分开编码，适合大规模召回；cross-encoder 同时读取 query 与文档，交互更充分但成本高，适合 top-N rerank。

下面用可解释特征模拟 reranker：query term coverage、编号/数字精确匹配、来源权威性、版本与 RRF 基线。它不是学习型 cross-encoder，但能展示接口、特征 trace 与硬规则边界。旧版本和无权文档已经在召回前过滤，不能靠 reranker“降权补救”。

In [ ]:
def exact_identifiers(text: str) -> set[str]:  # 定义本节可复用的核心函数。
    return set(re.findall(r"[a-z]+[0-9]+|[0-9]+(?:\.[0-9]+)?", text.lower()))  # 返回当前分支计算出的结果。

def rerank(query: str, fused: list[tuple[str, float]], rerank_n: int = 6):  # 定义本节可复用的核心函数。
    query_terms = set(analyze(query))  # 计算并保存当前步骤的中间状态。
    query_ids = exact_identifiers(query)  # 计算并保存当前步骤的中间状态。
    rows = []  # 计算并保存当前步骤的中间状态。
    for chunk_id, fused_score in fused[:rerank_n]:  # 遍历输入元素以累积或检查结果。
        chunk = by_id[chunk_id]  # 计算并保存当前步骤的中间状态。
        doc_terms = set(analyze(chunk.title + " " + chunk.text))  # 计算并保存当前步骤的中间状态。
        coverage = len(query_terms & doc_terms) / max(len(query_terms), 1)  # 计算并保存当前步骤的中间状态。
        exact = float(bool(query_ids) and query_ids <= exact_identifiers(chunk.title + " " + chunk.text))  # 计算并保存当前步骤的中间状态。
        score = 60 * fused_score + 1.4 * coverage + 1.0 * exact + 0.25 * chunk.authority + 0.03 * chunk.version  # 计算并保存当前步骤的中间状态。
        features = {"rrf_x60": 60*fused_score, "coverage": coverage, "exact_id": exact, "authority": chunk.authority, "version": chunk.version}  # 计算并保存当前步骤的中间状态。
        rows.append((chunk_id, score, features))  # 执行当前语句以推进本节示例。
    return sorted(rows, key=lambda row: (-row[1], row[0]))  # 返回当前分支计算出的结果。

fused, _ = hybrid_candidates("E1042 怎么处理", "company-a")  # 计算并保存当前步骤的中间状态。
for row in rerank("E1042 怎么处理", fused):  # 遍历输入元素以累积或检查结果。
    print(row)  # 执行当前语句以推进本节示例。


## 7. 评估必须覆盖召回、排序和分桶

- **Recall@k**：所有相关文档中有多少进入前 k，适合调候选深度。
- **MRR**：首个相关结果名次的倒数，适合“找到一个答案即可”的场景。
- **nDCG@k**：考虑多级相关性与位置折扣，适合多个证据质量不同的排序。

平均值必须按精确编号、自然语言改写、数字/时间、权限、无答案、多语言等分桶。没有 gold 的查询不能伪装成 Recall=1；应单独评估误召回、拒答和空结果体验。

In [ ]:
qrels = {  # 计算并保存当前步骤的中间状态。
    "E1042 怎么处理": {"db-e1042-v2#0": 3},  # 执行当前语句以推进本节示例。
    "数据库连不上怎么办": {"db-e1042-v2#0": 3, "slow-sql-v1#0": 1},  # 执行当前语句以推进本节示例。
    "上海酒店报销上限": {"travel-v2#0": 3},  # 执行当前语句以推进本节示例。
    "退款多久到账": {"refund-v1#0": 3},  # 执行当前语句以推进本节示例。
    "忘记密码无法登录": {"login-v1#0": 3},  # 执行当前语句以推进本节示例。
    "订单怎么取消": {"cancel-v1#0": 3},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def recall_at_k(ranked: list[str], relevance: dict[str, int], k: int) -> float:  # 定义本节可复用的核心函数。
    relevant = {doc_id for doc_id, grade in relevance.items() if grade > 0}  # 计算并保存当前步骤的中间状态。
    if not relevant:  # 按当前条件选择后续控制路径。
        raise ValueError("Recall 需要至少一个 gold；无答案查询应单独评估")  # 遇到非法合同立即显式失败。
    return len(set(ranked[:k]) & relevant) / len(relevant)  # 返回当前分支计算出的结果。

def reciprocal_rank(ranked: list[str], relevance: dict[str, int]) -> float:  # 定义本节可复用的核心函数。
    for rank, doc_id in enumerate(ranked, start=1):  # 遍历输入元素以累积或检查结果。
        if relevance.get(doc_id, 0) > 0:  # 按当前条件选择后续控制路径。
            return 1 / rank  # 返回当前分支计算出的结果。
    return 0.0  # 返回当前分支计算出的结果。

def ndcg_at_k(ranked: list[str], relevance: dict[str, int], k: int) -> float:  # 定义本节可复用的核心函数。
    def dcg(grades):  # 定义本节可复用的核心函数。
        return sum((2**grade - 1) / math.log2(rank + 1) for rank, grade in enumerate(grades, start=1))  # 返回当前分支计算出的结果。
    actual = [relevance.get(doc_id, 0) for doc_id in ranked[:k]]  # 计算并保存当前步骤的中间状态。
    ideal = sorted(relevance.values(), reverse=True)[:k]  # 计算并保存当前步骤的中间状态。
    denominator = dcg(ideal)  # 计算并保存当前步骤的中间状态。
    return dcg(actual) / denominator if denominator else 0.0  # 返回当前分支计算出的结果。

def ids_bm25(query): return [cid for cid, _ in bm25.search(query, "company-a", 6)]  # 定义本节可复用的核心函数。
def ids_dense(query): return [cid for cid, _ in dense_search(query, "company-a", 6)]  # 定义本节可复用的核心函数。
def ids_rrf(query): return [cid for cid, _ in hybrid_candidates(query, "company-a", 6, 6)[0]]  # 定义本节可复用的核心函数。
def ids_rerank(query): return [cid for cid, _, _ in rerank(query, hybrid_candidates(query, "company-a", 6, 6)[0])]  # 定义本节可复用的核心函数。

def evaluate(strategy):  # 定义本节可复用的核心函数。
    rows=[]  # 计算并保存当前步骤的中间状态。
    for query, relevance in qrels.items():  # 遍历输入元素以累积或检查结果。
        ranked = strategy(query)  # 计算并保存当前步骤的中间状态。
        rows.append((recall_at_k(ranked,relevance,3), reciprocal_rank(ranked,relevance), ndcg_at_k(ranked,relevance,3)))  # 执行当前语句以推进本节示例。
    return np.mean(rows, axis=0)  # 返回当前分支计算出的结果。

for name, strategy in [("BM25",ids_bm25),("LSA",ids_dense),("RRF",ids_rrf),("RRF+rerank",ids_rerank)]:  # 遍历输入元素以累积或检查结果。
    recall, mrr, ndcg = evaluate(strategy)  # 计算并保存当前步骤的中间状态。
    print(f"{name:12s} Recall@3={recall:.3f} MRR={mrr:.3f} nDCG@3={ndcg:.3f}")  # 计算并保存当前步骤的中间状态。


### 7.1 相关性指标之外，还要评估无答案误召回

上面的受控正例很容易，多个策略都可能得到 1.000；这只能说明指标实现和基本链路可运行，不能证明融合带来线上收益。回归集必须加入无答案、越权、旧版本和冲突证据。对无答案查询，本例报告 `false_positive_rate`；生产 RAG 还应报告拒答准确率、误拒率与回答忠实度，并分别调 retrieval threshold 和 answerability threshold。

In [ ]:
def precision_at_k(ranked: list[str], relevance: dict[str, int], k: int) -> float:  # 定义本节可复用的核心函数。
    if k <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("k 必须为正数")  # 遇到非法合同立即显式失败。
    return sum(relevance.get(doc_id, 0) > 0 for doc_id in ranked[:k]) / k  # 返回当前分支计算出的结果。

positive_precision = np.mean([  # 计算并保存当前步骤的中间状态。
    precision_at_k(ids_rerank(query), relevance, 1)  # 执行当前语句以推进本节示例。
    for query, relevance in qrels.items()  # 遍历输入元素以累积或检查结果。
])  # 执行当前语句以推进本节示例。
no_answer_queries = ["火星量子香蕉", "天气如何预测"]  # 计算并保存当前步骤的中间状态。
negative_outputs = {  # 计算并保存当前步骤的中间状态。
    query: {  # 执行当前语句以推进本节示例。
        "bm25": ids_bm25(query),  # 执行当前语句以推进本节示例。
        "dense": ids_dense(query),  # 执行当前语句以推进本节示例。
        "hybrid": ids_rrf(query),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    for query in no_answer_queries  # 遍历输入元素以累积或检查结果。
}  # 执行当前语句以推进本节示例。
false_positive_rate = np.mean([  # 计算并保存当前步骤的中间状态。
    bool(ranked)  # 执行当前语句以推进本节示例。
    for outputs in negative_outputs.values()  # 遍历输入元素以累积或检查结果。
    for ranked in outputs.values()  # 遍历输入元素以累积或检查结果。
])  # 执行当前语句以推进本节示例。
print({"positive_P@1": positive_precision, "no_answer_false_positive_rate": false_positive_rate})  # 执行当前语句以推进本节示例。
print(negative_outputs)  # 执行当前语句以推进本节示例。
assert false_positive_rate == 0.0  # 用受控断言验证关键不变量。


## 8. 候选深度是质量—延迟预算

先让 sparse/dense 各自取得足够 Recall，再看融合后的 union；最后限制昂贵 reranker 的候选数。若 top-N 从 20 提到 200 只增加重复候选，却让重排延迟放大 10 倍，就不是合理配置。

应画出 `Recall@N、候选去重率、reranker P95、最终 nDCG` 随 N 的曲线，并按查询类型选择预算。精确错误码可能只需 sparse 小候选；模糊自然语言问题需要更深 dense 或 query rewrite。

In [ ]:
def hybrid_ranked(query: str, depth: int) -> list[str]:  # 定义本节可复用的核心函数。
    return [cid for cid, _ in hybrid_candidates(query, "company-a", depth, depth)[0]]  # 返回当前分支计算出的结果。

for depth in [1, 2, 3, 5]:  # 遍历输入元素以累积或检查结果。
    recalls=[]  # 计算并保存当前步骤的中间状态。
    unions=[]  # 计算并保存当前步骤的中间状态。
    for query, relevance in qrels.items():  # 遍历输入元素以累积或检查结果。
        ranked=hybrid_ranked(query, depth)  # 计算并保存当前步骤的中间状态。
        recalls.append(recall_at_k(ranked, relevance, min(len(ranked), 2*depth)))  # 执行当前语句以推进本节示例。
        unions.append(len(ranked))  # 执行当前语句以推进本节示例。
    print(f"每路 depth={depth}: mean recall={np.mean(recalls):.3f}, mean union={np.mean(unions):.1f}")  # 计算并保存当前步骤的中间状态。


## 9. Trace、超时与降级

每次请求应记录 analyzer/index/model 版本、过滤条件、各路 raw rank/score、融合贡献、rerank 特征/模型版本、最终 context ids、每阶段耗时和降级原因。不要把全部敏感正文写入日志；稳定 id 足以回放。

建议为每路设置独立 deadline：dense 超时时可退回 BM25，reranker 超时可返回融合顺序，但响应和指标必须带 `degraded=true`。若 ACL 服务或过滤不可用，应 fail closed，不能为了可用性返回未过滤候选。

In [ ]:
def search_with_trace(query: str, tenant: str, dense_available=True, reranker_available=True):  # 定义本节可复用的核心函数。
    started=time.perf_counter()  # 计算并保存当前步骤的中间状态。
    sparse=bm25.search(query,tenant,4)  # 计算并保存当前步骤的中间状态。
    dense=dense_search(query,tenant,4) if dense_available else []  # 计算并保存当前步骤的中间状态。
    rankings={"bm25":sparse}  # 计算并保存当前步骤的中间状态。
    degraded=[]  # 计算并保存当前步骤的中间状态。
    if dense_available:  # 按当前条件选择后续控制路径。
        rankings["dense"]=dense  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        degraded.append("dense_unavailable")  # 执行当前语句以推进本节示例。
    fused, fusion=rrf(rankings)  # 计算并保存当前步骤的中间状态。
    if reranker_available:  # 按当前条件选择后续控制路径。
        final=[cid for cid,_,_ in rerank(query,fused)[:3]]  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        degraded.append("reranker_unavailable")  # 执行当前语句以推进本节示例。
        final=[cid for cid,_ in fused[:3]]  # 计算并保存当前步骤的中间状态。
    trace={  # 计算并保存当前步骤的中间状态。
        "query_tokens":analyze(query), "tenant":tenant, "filters":{"active":True},  # 执行当前语句以推进本节示例。
        "sparse":sparse, "dense":dense, "fusion":fusion, "final_ids":final,  # 执行当前语句以推进本节示例。
        "degraded":degraded, "elapsed_ms":(time.perf_counter()-started)*1000,  # 执行当前语句以推进本节示例。
        "versions":{"analyzer":"demo-v1","sparse_index":"2026-07","dense_model":"lsa-demo-v1"},  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return final,trace  # 返回当前分支计算出的结果。

final_ids,trace=search_with_trace("E1042 怎么处理","company-a",dense_available=False)  # 计算并保存当前步骤的中间状态。
print("final:",final_ids)  # 执行当前语句以推进本节示例。
print("degraded:",trace["degraded"],"elapsed_ms:",round(trace["elapsed_ms"],3))  # 执行当前语句以推进本节示例。


### 9.1 对外 API：可信身份、版本、预算与安全 trace

上一个函数是检索核心的教学 helper；对外边界还必须做四件事：只接受认证中间件生成的 context、校验目标索引版本与过滤器、限制 top-k/故障注入、移除可还原查询内容的日志字段。下面的 `search_api` 用 demo session 模拟认证边界，并用 fault set 模拟阶段超时；它不是在伪装真实 wall-clock 超时，而是给超时测试一个可重复注入点。真实服务应由 deadline/cancellation 触发同样的降级状态。

In [ ]:
from dataclasses import dataclass, field  # 导入本单元所需的依赖。

_AUTH_ISSUER_MARKER = object()  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class VerifiedContext:  # 定义承载本节状态与行为的数据结构。
    subject_id: str  # 执行当前语句以推进本节示例。
    tenant: str  # 执行当前语句以推进本节示例。
    authn_method: str  # 执行当前语句以推进本节示例。
    _issuer_marker: object = field(repr=False, compare=False)  # 计算并保存当前步骤的中间状态。

def authenticate_demo(session_id: str) -> VerifiedContext:  # 定义本节可复用的核心函数。
    # 真实系统由网关验证签名、issuer、audience、expiry，再注入不可由请求正文覆盖的 claims。
    if session_id != "signed-demo-session-a":  # 按当前条件选择后续控制路径。
        raise PermissionError("认证失败")  # 遇到非法合同立即显式失败。
    return VerifiedContext("user-17", INDEX_TENANT, "demo-signed-session", _AUTH_ISSUER_MARKER)  # 返回当前分支计算出的结果。

DEMO_CONTEXT = authenticate_demo("signed-demo-session-a")  # 计算并保存当前步骤的中间状态。

def search_api(  # 定义本节可复用的核心函数。
    query: str,  # 执行当前语句以推进本节示例。
    context: VerifiedContext,  # 执行当前语句以推进本节示例。
    *,  # 执行当前语句以推进本节示例。
    top_k: int = 3,  # 计算并保存当前步骤的中间状态。
    filters: dict | None = None,  # 计算并保存当前步骤的中间状态。
    index_version: str = INDEX_VERSION,  # 计算并保存当前步骤的中间状态。
    faults: frozenset[str] = frozenset(),  # 计算并保存当前步骤的中间状态。
):  # 执行当前语句以推进本节示例。
    if (not isinstance(context, VerifiedContext) or  # 按当前条件选择后续控制路径。
            context._issuer_marker is not _AUTH_ISSUER_MARKER or  # 执行当前语句以推进本节示例。
            context.tenant != INDEX_TENANT):  # 计算并保存当前步骤的中间状态。
        raise PermissionError("tenant 必须来自已验证且绑定当前索引的身份上下文")  # 遇到非法合同立即显式失败。
    if not 1 <= top_k <= 3:  # 按当前条件选择后续控制路径。
        raise ValueError("教学服务 top_k 必须在 1..3")  # 遇到非法合同立即显式失败。
    if index_version != INDEX_VERSION:  # 按当前条件选择后续控制路径。
        raise ValueError("请求的 index_version 与已加载快照不一致")  # 遇到非法合同立即显式失败。
    if filters is None:  # 按当前条件选择后续控制路径。
        filters = {"active": True}  # 计算并保存当前步骤的中间状态。
    if filters != {"active": True}:  # 按当前条件选择后续控制路径。
        raise ValueError("本索引只支持固定 active filter")  # 遇到非法合同立即显式失败。
    allowed_faults = {"dense_timeout", "reranker_timeout"}  # 计算并保存当前步骤的中间状态。
    if not faults <= allowed_faults:  # 按当前条件选择后续控制路径。
        raise ValueError("未知故障注入")  # 遇到非法合同立即显式失败。

    final, internal_trace = search_with_trace(  # 计算并保存当前步骤的中间状态。
        query, context.tenant,  # 执行当前语句以推进本节示例。
        dense_available="dense_timeout" not in faults,  # 计算并保存当前步骤的中间状态。
        reranker_available="reranker_timeout" not in faults,  # 计算并保存当前步骤的中间状态。
    )  # 执行当前语句以推进本节示例。
    # raw ranks/scores/candidate IDs 仅进入受控内部诊断存储，不放进普通 API trace。
    trace = {  # 计算并保存当前步骤的中间状态。
        "query_logging": "redacted",  # 执行当前语句以推进本节示例。
        "query_length": len(query),  # 执行当前语句以推进本节示例。
        "analyzed_token_count": len(analyze(query)),  # 执行当前语句以推进本节示例。
        "authn_method": context.authn_method,  # 执行当前语句以推进本节示例。
        "index_version": index_version,  # 执行当前语句以推进本节示例。
        "requested_top_k": top_k,  # 执行当前语句以推进本节示例。
        "degraded": internal_trace["degraded"],  # 执行当前语句以推进本节示例。
        "elapsed_ms": internal_trace["elapsed_ms"],  # 执行当前语句以推进本节示例。
        "fault_injection": sorted(faults),  # 执行当前语句以推进本节示例。
        "stage_deadline_ms": {"sparse": 30, "dense": 50, "reranker": 40},  # 执行当前语句以推进本节示例。
        "debug_trace_exported": False,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return final[:top_k], trace  # 返回当前分支计算出的结果。

api_ids, api_trace = search_api("E1042 怎么处理", DEMO_CONTEXT, top_k=2)  # 计算并保存当前步骤的中间状态。
print(api_ids, {key: api_trace[key] for key in ["index_version", "degraded", "query_logging"]})  # 执行当前语句以推进本节示例。


## 10. 工程断言比漂亮 demo 更重要

最低测试集应覆盖：同一请求确定性、精确编号召回、无权文档不进入任何候选、inactive 旧版本被过滤、单路降级仍可返回、所有结果 id 可追溯。生产回归还应加入索引升级对比、长尾查询、拼写错误、冲突文档、空 gold 和延迟故障注入。

In [ ]:
first,first_trace=search_api("E1042 怎么处理",DEMO_CONTEXT)  # 计算并保存当前步骤的中间状态。
second,_=search_api("E1042 怎么处理",DEMO_CONTEXT)  # 计算并保存当前步骤的中间状态。
assert first==second  # 用受控断言验证关键不变量。
assert first[0]=="db-e1042-v2#0"  # 用受控断言验证关键不变量。
assert "tenant-b-secret#0" not in by_id and "tenant-b-secret#0" not in first  # 用受控断言验证关键不变量。
assert "travel-v1#0" not in by_id and "travel-v1#0" not in ids_rrf("上海酒店报销上限")  # 用受控断言验证关键不变量。
assert bm25.search("火星量子香蕉", INDEX_TENANT, 3) == []  # 用受控断言验证关键不变量。
assert search_api("火星量子香蕉", DEMO_CONTEXT)[0] == []  # 用受控断言验证关键不变量。
assert "query_tokens" not in first_trace and first_trace["debug_trace_exported"] is False  # 用受控断言验证关键不变量。
fallback,fallback_trace=search_api("退款多久到账",DEMO_CONTEXT,faults=frozenset({"dense_timeout","reranker_timeout"}))  # 计算并保存当前步骤的中间状态。
assert fallback and fallback_trace["degraded"]==["dense_unavailable","reranker_unavailable"]  # 用受控断言验证关键不变量。
assert all(chunk_id in by_id for chunk_id in fallback)  # 用受控断言验证关键不变量。
for invalid_context in ["company-b", VerifiedContext("attacker", INDEX_TENANT, "forged", object())]:  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        search_api("E1042", invalid_context)  # 执行当前语句以推进本节示例。
        raise AssertionError("伪造身份不应被接受")  # 遇到非法合同立即显式失败。
    except PermissionError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。
try:  # 尝试执行可能失败的受控操作。
    search_api("E1042", DEMO_CONTEXT, filters={})  # 计算并保存当前步骤的中间状态。
    raise AssertionError("显式空 filters 不应被当作默认值")  # 遇到非法合同立即显式失败。
except ValueError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
print("ALL ENGINEERING ASSERTIONS PASSED")  # 执行当前语句以推进本节示例。


## 11. 生产检查表

### 离线

- qrels 是否覆盖关键查询分桶，标注协议和 inter-annotator agreement 是否记录？
- sparse analyzer、dense model、chunk 与 index 是否绑定同一发布版本？
- Recall@N、MRR/nDCG、ACL、旧版本和空答案测试是否都通过？
- 索引升级能否双写、回放、灰度、回滚？

### 在线

- 每阶段 deadline、熔断和降级是否显式进入 trace/指标？
- 是否监控候选 union、去重率、各路贡献、rerank 改排率与 zero-result rate？
- P50/P95/P99、单位查询成本和缓存键是否按 tenant/权限/版本隔离？
- 用户点击只是一种偏置信号，不能直接等同相关性标签。

### 面试回答模板

先说明 sparse 保精确词、dense 补语义改写；两路先做权限过滤并取各自 top-N，再用 RRF 建立不依赖分数尺度的基线，最后用较贵 reranker 精排。用 Recall@N 调候选深度，用 MRR/nDCG 调排序，同时记录完整 trace、超时降级和索引版本。

## 12. 研究依据与进一步阅读

- Robertson & Zaragoza, *The Probabilistic Relevance Framework: BM25 and Beyond*：BM25/BM25F 的概率相关框架与参数解释。https://doi.org/10.1561/1500000019
- Apache Lucene `BM25Similarity` 官方 API：IDF 形式、`k1`/`b` 约束与默认值。https://lucene.apache.org/core/9_11_1/core/org/apache/lucene/search/similarities/BM25Similarity.html
- Cormack, Clarke & Büttcher, *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods*：RRF 原始论文。https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
- Karpukhin et al., *Dense Passage Retrieval for Open-Domain Question Answering*：双编码 dense retrieval。https://arxiv.org/abs/2004.04906
- Nogueira & Cho, *Passage Re-ranking with BERT*：query-document 联合编码重排。https://arxiv.org/abs/1901.04085

本 notebook 的 LSA 和手工 reranker 是为了让整条链路离线可执行；真实系统必须替换为目标 embedding/cross-encoder，并在自己的 qrels、流量和硬件上重新验证。